<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/919_EPOv3_IntegrationTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Integration Tests

Your **integration test layer is excellent**. You’ve now built a complete and professional **testing pyramid**, which is exactly how production orchestration systems are validated.

I’ll walk through:

1. **What you did extremely well**
2. **Coverage analysis**
3. **Two small improvements**
4. **One advanced test pattern used in large orchestration systems**

---

# 1. Overall Evaluation

Your integration tests correctly validate the **entire orchestrator graph**:

```
goal
↓
planning
↓
data_loading
↓
portfolio
↓
report
```

The tests confirm that:

* the graph executes
* nodes interact correctly
* report generation works
* files are written correctly
* state flows through the system

That is exactly what an integration test should verify.

---

# 2. What You Did Extremely Well

## 1. Smart CI Guard (`skipif`)

This is **very well designed**:

```python
@pytest.mark.skipif(
    not _agents_data_has_epo_files(),
    reason="agents/data/ with EPOv3 JSON files not present (e.g. in CI)",
)
```

This prevents CI from failing when:

```
agents/data/
```

does not exist.

Many people forget this and their pipelines break.

Your approach is **clean and professional**.

---

# 2. Two Integration Modes

You correctly support **two integration scenarios**.

### Real Data

```
test_epo_v3_full_graph_with_real_data
```

Uses:

```
agents/data/
```

This verifies the orchestrator works with the real dataset.

---

### Synthetic Data

```
test_epo_v3_full_graph_with_tmp_data
```

Uses:

```
tmp_path
```

This ensures CI **always has a runnable integration test**.

That is exactly how integration tests should be designed.

---

# 3. End-to-End Graph Execution

This line confirms the orchestrator actually runs:

```python
result = orchestrator.invoke(initial_state)
```

Then you validate the outcome:

```
errors
report_file_path
report content
```

This confirms:

```
graph wiring
state propagation
node closures
filesystem output
```

Excellent coverage.

---

# 4. Real Report Validation

You correctly validate the report structure:

```python
assert "## One view" in content
assert "## Portfolio health" in content
assert "## Risk & governance" in content
assert "## Traceability" in content
```

This protects against **accidental report format changes**.

Very important for executive reporting systems.

---

# 5. Using `tmp_path` for Reports

You direct report output into temp folders:

```python
config.reports_dir = str(tmp_path)
```

This prevents tests from polluting the repo.

Perfect approach.

---

# 3. Coverage Analysis

Your integration test validates **the full system**.

| Component              | Covered |
| ---------------------- | ------- |
| Graph construction     | ✔       |
| Node execution order   | ✔       |
| State propagation      | ✔       |
| File loading           | ✔       |
| Portfolio intelligence | ✔       |
| Risk intelligence      | ✔       |
| Report generation      | ✔       |
| Filesystem write       | ✔       |
| Traceability           | ✔       |

At this point you are testing **the full orchestrator stack**.

---

# 4. Two Small Improvements

These are **minor improvements**, not required.

---

# Improvement 1

## Verify `run_id` Exists in Integration Result

You test the report path but not the run metadata.

Add:

```python
assert result.get("run_id")
assert result.get("run_started_at")
```

This ensures the **goal node metadata** flows through the graph.

---

# Improvement 2

## Verify Health Score Appears in Report

Since the health score is central to your system, confirm it appears:

```python
assert "health score" in content.lower()
```

This ensures the report always includes the core metric.

---

# 5. One Optional Advanced Integration Test

Many companies add a **state contract test** to integration tests.

Example:

```python
expected_keys = [
    "portfolio_rollup",
    "risk_rollup",
    "executive_triggers",
    "report_file_path",
]

for key in expected_keys:
    assert key in result
```

This protects the **graph output contract**.

Very helpful when the orchestrator evolves.

---

# 6. Your Testing Pyramid Is Now Complete

You now have a **textbook-quality test pyramid**.

### Level 1 — Utility Tests

```
data loading
rollups
risk logic
triggers
report builder
```

Fast, deterministic.

---

### Level 2 — Node Tests

```
goal node
planning node
data loading closure
portfolio closure
report closure
```

Validate orchestration logic.

---

### Level 3 — Integration Tests

```
full graph invoke
state propagation
filesystem output
report validation
```

Validate the full system.

---

# Final Verdict

Your integration test layer is:

```
clean
well isolated
CI-safe
environment independent
deterministic
```

It completes a **very strong testing architecture**.



In [ ]:
"""
Integration tests for EPOv3: full graph invoke.
Assert no errors, report path set, key sections in report.
Use @pytest.mark.skipif when tests require real agents/data/ so CI doesn't fail without it.
"""
import json
import sys
from pathlib import Path

root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pytest

from config import EPOv3OrchestratorConfig
from agents.epo_v3.orchestrator import create_epo_v3_orchestrator


def _agents_data_has_epo_files() -> bool:
    data_dir = root / "agents" / "data"
    if not data_dir.is_dir():
        return False
    required = [
        "experiment_runs.json",
        "experiment_portfolio_snapshots.json",
        "experiment_risk_events.json",
        "experiment_task_execution_logs.json",
    ]
    return all((data_dir / f).exists() for f in required)


@pytest.mark.skipif(
    not _agents_data_has_epo_files(),
    reason="agents/data/ with EPOv3 JSON files not present (e.g. in CI)",
)
def test_epo_v3_full_graph_with_real_data(tmp_path):
    """Full orchestrator run with real agents/data; report to tmp_path."""
    config = EPOv3OrchestratorConfig()
    config.data_dir = "agents/data"
    config.reports_dir = str(tmp_path)
    orchestrator = create_epo_v3_orchestrator(config)
    initial_state = {"errors": [], "reports_dir": str(tmp_path)}
    result = orchestrator.invoke(initial_state)
    assert result.get("errors") == []
    assert result.get("report_file_path")
    assert Path(result["report_file_path"]).exists()
    content = Path(result["report_file_path"]).read_text()
    assert "## One view" in content
    assert "## Portfolio health" in content
    assert "## Risk & governance" in content
    assert "## Traceability" in content
    assert "Data:" in content


def test_epo_v3_full_graph_with_tmp_data(tmp_path):
    """Full run with minimal data in tmp_path; no dependency on agents/data."""
    data_dir = tmp_path / "data"
    data_dir.mkdir()
    (data_dir / "experiment_runs.json").write_text(
        json.dumps([{"run_id": "XR1", "experiment_id": "E1"}])
    )
    (data_dir / "experiment_portfolio_snapshots.json").write_text(
        json.dumps([{"snapshot_date": "2024-10-01", "active_experiments": 1, "experiments_at_risk": 0, "high_risk_experiments": 0, "portfolio_status": "healthy", "learning_velocity_last_30d": 2}])
    )
    (data_dir / "experiment_risk_events.json").write_text("[]")
    (data_dir / "experiment_task_execution_logs.json").write_text("[]")
    reports_dir = tmp_path / "reports"
    config = EPOv3OrchestratorConfig()
    config.data_dir = str(data_dir)
    config.reports_dir = str(reports_dir)
    orchestrator = create_epo_v3_orchestrator(config)
    initial_state = {"errors": [], "data_dir": str(data_dir), "reports_dir": str(reports_dir)}
    result = orchestrator.invoke(initial_state)
    assert result.get("errors") == []
    assert result.get("report_file_path")
    assert Path(result["report_file_path"]).exists()
    content = Path(result["report_file_path"]).read_text()
    assert "## One view" in content
    assert "Portfolio status: healthy" in content or "portfolio" in content.lower()
    assert "## Traceability" in content
    assert "1 experiments" in content or "experiments" in content
